# Walmart Sales Forecasting
### Store Segregation · ARIMA · SARIMA · Gradient Boosting
Run cells in order. Upload `Walmart_DataSet.csv` when prompted.

## CELL 1 — Install Libraries

In [ ]:
# !pip install statsmodels --upgrade -q

## CELL 2 — Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import itertools
import joblib
import os

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

print("✅ All libraries imported successfully")

## CELL 3 — Upload & Load Dataset

In [ ]:
# ─────────────────────────────────────────────────────────
# DATASET LOADING — choose ONE option below
# ─────────────────────────────────────────────────────────

# OPTION A — Upload from your computer (recommended for Colab)
from google.colab import files
print("📂 Upload Walmart_DataSet.csv when the dialog appears...")
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

# OPTION B — Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Walmart_DataSet.csv')

# OPTION C — Direct path (if already uploaded or running locally)
# df = pd.read_csv("Walmart_DataSet.csv")

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

print(f"\n✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Stores     : {df['Store'].nunique()} unique stores")
print(f"   Date range : {df['Date'].min().date()}  →  {df['Date'].max().date()}")
print(f"   Weeks/store: {df.groupby('Store').size().mean():.0f}")
print(f"\n   Missing values:\n{df.isnull().sum().to_string()}")
print(f"   Duplicates : {df.duplicated().sum()}")
print(f"\n   First 5 rows:")
df.head()


## CELL 4 — Data Description & Statistics

In [ ]:
print("=" * 60)
print("DATASET STATISTICAL SUMMARY")
print("=" * 60)
print(df.describe().round(2).to_string())

print("\n\nDATA TYPES:")
print(df.dtypes.to_string())

## CELL 5 — STORE SEGREGATION (All 45 Stores)

In [ ]:
print("=" * 60)
print("STORE SEGREGATION — Profiling All 45 Stores")
print("=" * 60)

store_profile = df.groupby("Store").agg(
    Avg_Weekly_Sales  = ("Weekly_Sales", "mean"),
    Total_Sales       = ("Weekly_Sales", "sum"),
    Max_Weekly_Sales  = ("Weekly_Sales", "max"),
    Min_Weekly_Sales  = ("Weekly_Sales", "min"),
    Std_Weekly_Sales  = ("Weekly_Sales", "std"),
    Weeks             = ("Date",         "count"),
).round(2).reset_index()

# Assign tiers
store_profile["Sales_Tier"] = pd.qcut(
    store_profile["Avg_Weekly_Sales"],
    q=3, labels=["Low", "Mid", "High"]
)

print("\n  Store Sales Tier Summary:")
print(f"  HIGH tier stores : {store_profile[store_profile['Sales_Tier']=='High']['Store'].tolist()}")
print(f"  MID  tier stores : {store_profile[store_profile['Sales_Tier']=='Mid']['Store'].tolist()}")
print(f"  LOW  tier stores : {store_profile[store_profile['Sales_Tier']=='Low']['Store'].tolist()}")

print("\n  Top 10 stores by Average Weekly Sales:")
print(store_profile.nlargest(10, "Avg_Weekly_Sales")[
    ["Store","Avg_Weekly_Sales","Total_Sales","Max_Weekly_Sales","Sales_Tier"]
].to_string(index=False))

print("\n  Bottom 5 stores by Average Weekly Sales:")
print(store_profile.nsmallest(5, "Avg_Weekly_Sales")[
    ["Store","Avg_Weekly_Sales","Total_Sales","Max_Weekly_Sales","Sales_Tier"]
].to_string(index=False))

store_profile.head(10)

## CELL 6 — EDA: Sales Distribution & Store Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Weekly sales distribution
axes[0].hist(df["Weekly_Sales"] / 1e6, bins=50,
             color="#0071CE", edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Weekly Sales (Millions $)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Weekly Sales — All Stores")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}M"))

# 2. Avg sales by store (sorted)
sp_sorted = store_profile.sort_values("Avg_Weekly_Sales")
bar_colors = ["#22C55E" if t == "High" else "#3B82F6" if t == "Mid" else "#F97316"
              for t in sp_sorted["Sales_Tier"]]
axes[1].barh(sp_sorted["Store"].astype(str),
             sp_sorted["Avg_Weekly_Sales"] / 1e6, color=bar_colors)
axes[1].set_xlabel("Avg Weekly Sales (M$)")
axes[1].set_title("All 45 Stores — Ranked by Avg Sales\n(Green=High, Blue=Mid, Orange=Low)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}M"))

plt.tight_layout()
plt.suptitle("Store Segregation — Sales Distribution", fontsize=14,
             fontweight="bold", y=1.01)
plt.show()

## CELL 7 — EDA: Sales Heatmap All 45 Stores

In [ ]:
pivot = df.pivot_table(
    values="Weekly_Sales",
    index="Store",
    columns=df["Date"].dt.to_period("Q").astype(str),
    aggfunc="mean"
)

fig, ax = plt.subplots(figsize=(18, 11))
sns.heatmap(pivot / 1e6, cmap="YlOrRd", ax=ax,
            linewidths=0.3, linecolor="#eee",
            cbar_kws={"label": "Avg Weekly Sales (M$)"})
ax.set_title("Weekly Sales Heatmap — All 45 Stores by Quarter",
             fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Quarter", fontsize=11)
ax.set_ylabel("Store Number", fontsize=11)
plt.tight_layout()
plt.show()

## CELL 8 — EDA: Seasonality, Holiday, Macro Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Monthly seasonality
monthly = df.groupby(df["Date"].dt.month)["Weekly_Sales"].mean()
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
axes[0,0].bar(month_labels, monthly.values / 1e6,
              color=sns.color_palette("Blues_d", 12))
axes[0,0].set_title("Average Sales by Month (Seasonality)")
axes[0,0].set_ylabel("Avg Weekly Sales (M$)")
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.2f}M"))

# 2. Holiday vs Non-Holiday
holiday_avg    = df[df["Holiday_Flag"] == 1]["Weekly_Sales"].mean()
nonholiday_avg = df[df["Holiday_Flag"] == 0]["Weekly_Sales"].mean()
axes[0,1].bar(["Non-Holiday", "Holiday"],
              [nonholiday_avg / 1e6, holiday_avg / 1e6],
              color=["#3B82F6", "#F97316"], width=0.5)
axes[0,1].set_title(f"Holiday vs Non-Holiday Sales\nHoliday premium: +{(holiday_avg/nonholiday_avg-1)*100:.1f}%")
axes[0,1].set_ylabel("Avg Weekly Sales (M$)")
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.2f}M"))
for i, v in enumerate([nonholiday_avg, holiday_avg]):
    axes[0,1].text(i, v/1e6 + 0.01, f"${v/1e6:.3f}M", ha="center", fontweight="bold")

# 3. Temperature vs Sales
axes[1,0].scatter(df["Temperature"], df["Weekly_Sales"]/1e6,
                  alpha=0.15, s=8, c=df["Holiday_Flag"], cmap="bwr")
axes[1,0].set_title("Temperature vs Weekly Sales (Red=Holiday)")
axes[1,0].set_xlabel("Temperature (°F)")
axes[1,0].set_ylabel("Weekly Sales (M$)")
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.1f}M"))

# 4. Unemployment vs Sales
axes[1,1].scatter(df["Unemployment"], df["Weekly_Sales"]/1e6,
                  alpha=0.15, s=8, color="#8B5CF6")
axes[1,1].set_title("Unemployment Rate vs Weekly Sales")
axes[1,1].set_xlabel("Unemployment Rate (%)")
axes[1,1].set_ylabel("Weekly Sales (M$)")
axes[1,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.1f}M"))

plt.suptitle("Exploratory Data Analysis", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## CELL 9 — Correlation Heatmap

In [ ]:
corr_df = df[["Weekly_Sales","Holiday_Flag","Temperature","Fuel_Price",
              "CPI","Unemployment"]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, annot_kws={"size": 10})
ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## CELL 10 — Select 5 Focus Stores for SARIMA

In [ ]:
# Choose top-2, 1 mid, bottom-2 for variety
top2    = store_profile.nlargest(2, "Avg_Weekly_Sales")["Store"].tolist()
mid1    = [store_profile.sort_values("Avg_Weekly_Sales").iloc[len(store_profile)//2]["Store"]]
bottom2 = store_profile.nsmallest(2, "Avg_Weekly_Sales")["Store"].tolist()
FOCUS_STORES = top2 + mid1 + bottom2
FOCUS_STORES = [int(s) for s in FOCUS_STORES]

STORE_COLORS = {
    FOCUS_STORES[0]: "#2ECC71",
    FOCUS_STORES[1]: "#3B82F6",
    FOCUS_STORES[2]: "#F97316",
    FOCUS_STORES[3]: "#8B5CF6",
    FOCUS_STORES[4]: "#EF4444",
}

print("=" * 60)
print("FOCUS STORES SELECTED FOR SARIMA DEEP-DIVE")
print("=" * 60)
for s in FOCUS_STORES:
    row = store_profile[store_profile["Store"] == s].iloc[0]
    print(f"  Store {s:2d} | Tier: {row['Sales_Tier']:4s} | "
          f"Avg: ${row['Avg_Weekly_Sales']:>12,.0f}/wk | "
          f"Total: ${row['Total_Sales']/1e6:.1f}M")

# Historical overlay for focus stores
fig, ax = plt.subplots(figsize=(14, 5))
for store in FOCUS_STORES:
    ts = df[df["Store"] == store].sort_values("Date")
    tier = store_profile[store_profile["Store"]==store]["Sales_Tier"].values[0]
    ax.plot(ts["Date"], ts["Weekly_Sales"]/1e6,
            lw=1.8, label=f"Store {store} ({tier})",
            color=STORE_COLORS[store])
ax.set_title("Historical Weekly Sales — 5 Focus Stores (Full Range)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Weekly Sales (M$)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.1f}M"))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## CELL 11 — Stationarity Tests (ADF) for 5 Stores

In [ ]:
print("=" * 60)
print("AUGMENTED DICKEY-FULLER STATIONARITY TEST")
print("H0: Series has a unit root (non-stationary)")
print("Reject H0 if p-value < 0.05  →  Series IS stationary")
print("=" * 60)

adf_results = {}
for store in FOCUS_STORES:
    ts = df[df["Store"] == store].set_index("Date")["Weekly_Sales"]
    stat, p, lags, nobs, crit, _ = adfuller(ts)
    status = "✅ STATIONARY"  if p < 0.05 else "⚠️  NON-STATIONARY (d=1 needed)"
    adf_results[store] = {"stat": stat, "p": p, "stationary": p < 0.05}
    print(f"\n  Store {store:2d}:")
    print(f"    ADF Statistic : {stat:.4f}")
    print(f"    p-value       : {p:.6f}")
    print(f"    Critical 5%   : {crit['5%']:.4f}")
    print(f"    Result        : {status}")

## CELL 12 — ACF & PACF Plots (Store 20 example)

In [ ]:
ref_store = FOCUS_STORES[0]
ts_ref = df[df["Store"] == ref_store].sort_values("Date").set_index("Date")["Weekly_Sales"]

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
plot_acf(ts_ref, lags=52, ax=axes[0],
         title=f"ACF — Store {ref_store} Weekly Sales (lags=52 weeks)")
plot_pacf(ts_ref, lags=52, ax=axes[1],
          title=f"PACF — Store {ref_store} Weekly Sales",
          method="ywm")
plt.suptitle("Autocorrelation Analysis — Guides ARIMA Order Selection",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  ACF: significant spikes at lag 52 → annual seasonality confirmed")
print("  PACF: cut-off after lag 1–2 → AR order p=1 or 2 recommended")

## CELL 13 — ARIMA: Auto-select Best Order via AIC

In [ ]:
def fit_best_arima(ts, max_p=3, max_q=3):
    """Grid-search ARIMA(p,1,q) and return best by AIC."""
    best_aic, best_order, best_fit = np.inf, (1,1,1), None
    for p, q in itertools.product(range(max_p+1), range(max_q+1)):
        try:
            m = ARIMA(ts, order=(p, 1, q)).fit()
            if m.aic < best_aic:
                best_aic, best_order, best_fit = m.aic, (p,1,q), m
        except Exception:
            pass
    return best_fit, best_order, best_aic

print("=" * 60)
print("ARIMA — Auto Order Selection (AIC Grid Search)")
print("Testing orders: p ∈ {0..3}, d=1 (differencing), q ∈ {0..3}")
print("=" * 60)

arima_results = {}
for store in FOCUS_STORES:
    ts = df[df["Store"] == store].sort_values("Date").set_index("Date")["Weekly_Sales"].copy()
    n = len(ts)
    split = max(int(n * 0.85), n - 20)
    split = min(split, n - 5)  # always keep ≥ 5 test weeks
    train = ts.iloc[:split]
    test  = ts.iloc[split:]

    try:
        fit, order, aic = fit_best_arima(train, max_p=3, max_q=3)
        if fit is None:
            print(f"  Store {store:2d}: ARIMA grid search failed — skipping")
            continue
        fc   = fit.forecast(steps=len(test))
        rmse = np.sqrt(mean_squared_error(test.values, fc.values))
        mae  = mean_absolute_error(test.values, fc.values)
        r2   = r2_score(test.values, fc.values)
        arima_results[store] = {
            "order": order, "aic": round(aic,2),
            "rmse": round(rmse,2), "mae": round(mae,2), "r2": round(r2,4),
            "fit": fit, "test": test, "fc": fc
        }
        tier = store_profile[store_profile["Store"]==store]["Sales_Tier"].values[0]
        print(f"\n  Store {store:2d} ({tier}) → Best ARIMA{order}")
        print(f"    AIC  : {aic:.2f}")
        print(f"    RMSE : ${rmse:>10,.2f}")
        print(f"    MAE  : ${mae:>10,.2f}")
        print(f"    R²   : {r2:.4f}")
    except Exception as e:
        print(f"  Store {store:2d}: ARIMA failed — {e}")
        continue


## CELL 14 — SARIMA: Fit for All 5 Focus Stores

In [ ]:
# SARIMA(p,d,q)(P,D,Q,s)
#   p=1, d=1, q=1   → non-seasonal AR, differencing, MA
#   P=1, D=1, Q=0, s=52 → seasonal period = 52 weeks (annual)

print("=" * 60)
print("SARIMA — Order: (1,1,1)(1,1,0,52)")
print("Seasonal period s=52 captures annual weekly pattern")
print("=" * 60)

sarima_results = {}
for store in FOCUS_STORES:
    ts = df[df["Store"] == store].sort_values("Date").set_index("Date")["Weekly_Sales"].copy()
    ts.index.freq = "W-FRI"
    n = len(ts)
    split = max(int(n * 0.85), n - 20)
    split = min(split, n - 5)
    train = ts.iloc[:split]
    test  = ts.iloc[split:]

    # Try SARIMA(1,1,1)(1,1,0,52) first; fallback to (1,1,1)(1,0,0,52) if needed
    fitted = None
    model_used = ""
    for seasonal_order, label in [
        ((1,1,0,52), "SARIMA(1,1,1)(1,1,0,52)"),
        ((1,0,0,52), "SARIMA(1,1,1)(1,0,0,52)"),
        ((0,0,0, 0), "ARIMA(1,1,1) fallback"),
    ]:
        try:
            if seasonal_order[3] > 0:
                m = SARIMAX(train,
                            order=(1,1,1),
                            seasonal_order=seasonal_order,
                            enforce_stationarity=False,
                            enforce_invertibility=False).fit(disp=False)
            else:
                m = ARIMA(train, order=(1,1,1)).fit()
            fitted = m
            model_used = label
            break
        except Exception as e:
            print(f"    ↳ {label} failed: {e}")
            continue

    if fitted is None:
        print(f"  Store {store:2d}: All SARIMA attempts failed — skipping")
        continue

    fc   = fitted.forecast(steps=len(test))
    rmse = np.sqrt(mean_squared_error(test.values, fc.values))
    mae  = mean_absolute_error(test.values, fc.values)
    r2   = r2_score(test.values, fc.values)

    sarima_results[store] = {
        "model_used": model_used,
        "aic":  round(fitted.aic, 2),
        "rmse": round(rmse, 2), "mae": round(mae, 2), "r2": round(r2, 4),
        "fit": fitted, "test": test, "fc": fc, "train": train
    }
    tier = store_profile[store_profile["Store"]==store]["Sales_Tier"].values[0]
    print(f"\n  Store {store:2d} ({tier}) → {model_used}")
    print(f"    AIC  : {fitted.aic:.2f}")
    print(f"    RMSE : ${rmse:>10,.2f}")
    print(f"    MAE  : ${mae:>10,.2f}")
    print(f"    R²   : {r2:.4f}")


## CELL 15 — ARIMA vs SARIMA Comparison Chart

In [ ]:
# Rebuild common list from successful fits
common = [s for s in FOCUS_STORES if s in arima_results and s in sarima_results]
print(f'  Stores with both ARIMA & SARIMA results: {common}')

store_labels = [f"Store {s}" for s in FOCUS_STORES]
arima_rmse  = [arima_results[s]["rmse"]/1e3 for s in FOCUS_STORES]
sarima_rmse = [sarima_results[s]["rmse"]/1e3 for s in FOCUS_STORES]
arima_r2    = [arima_results[s]["r2"]  for s in FOCUS_STORES]
sarima_r2   = [sarima_results[s]["r2"] for s in FOCUS_STORES]

x = np.arange(len(FOCUS_STORES))
w = 0.35

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# RMSE
axes[0].bar(x - w/2, arima_rmse,  w, label="ARIMA",  color="#94A3B8", alpha=0.9)
axes[0].bar(x + w/2, sarima_rmse, w, label="SARIMA", color="#22C55E", alpha=0.9)
axes[0].set_xticks(x); axes[0].set_xticklabels(store_labels, rotation=15)
axes[0].set_title("RMSE Comparison (K$)\n(Lower = Better)")
axes[0].set_ylabel("RMSE (K$)")
axes[0].legend()

# R²
axes[1].bar(x - w/2, arima_r2,  w, label="ARIMA",  color="#94A3B8", alpha=0.9)
axes[1].bar(x + w/2, sarima_r2, w, label="SARIMA", color="#3B82F6", alpha=0.9)
axes[1].set_xticks(x); axes[1].set_xticklabels(store_labels, rotation=15)
axes[1].set_title("R² Score Comparison\n(Higher = Better)")
axes[1].set_ylabel("R²")
axes[1].legend()

# Winner count
sarima_wins = sum(1 for s in FOCUS_STORES
                  if sarima_results[s]["rmse"] < arima_results[s]["rmse"])
arima_wins  = len(FOCUS_STORES) - sarima_wins
axes[2].bar(["ARIMA Wins", "SARIMA Wins"],
            [arima_wins, sarima_wins],
            color=["#94A3B8", "#22C55E"], alpha=0.9, width=0.4)
axes[2].set_title(f"Model Win Count\n(SARIMA wins {sarima_wins}/{len(FOCUS_STORES)} stores)")
axes[2].set_ylabel("Count")
for i, v in enumerate([arima_wins, sarima_wins]):
    axes[2].text(i, v + 0.05, str(v), ha="center", fontsize=14, fontweight="bold")

plt.suptitle("ARIMA vs SARIMA — Performance Comparison (5 Focus Stores)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 75)
print(f"{'Store':>8} | {'Tier':>4} | {'ARIMA Order':>12} | {'ARIMA R²':>9} | {'SARIMA R²':>9} | {'SARIMA RMSE':>12} | Winner")
print("-" * 75)
for s in FOCUS_STORES:
    a = arima_results[s]
    sr = sarima_results[s]
    tier = store_profile[store_profile["Store"]==s]["Sales_Tier"].values[0]
    winner = "SARIMA ✓" if sr["rmse"] < a["rmse"] else "ARIMA  ✓"
    print(f"  Store {s:2d} | {tier:>4} | {str(a['order']):>12} | {a['r2']:>9.4f} | {sr['r2']:>9.4f} | {sr['rmse']:>12,.0f} | {winner}")

## CELL 16 — SARIMA Test Set Fit Plot (5 stores)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 20), sharex=False)
fig.suptitle("SARIMA — Test Set Fit (Last 15% of data)\nvs Actual Sales",
             fontsize=14, fontweight="bold", y=1.01)

for idx, store in enumerate(FOCUS_STORES):
    a  = sarima_results[store]
    ax = axes[idx]
    color = list(STORE_COLORS.values())[idx]
    tier  = store_profile[store_profile["Store"]==store]["Sales_Tier"].values[0]

    ax.plot(a["train"].index, a["train"].values/1e6,
            color="steelblue", lw=1.5, alpha=0.7, label="Train")
    ax.plot(a["test"].index,  a["test"].values/1e6,
            color="black",     lw=2.0, label="Actual Test")
    ax.plot(a["fc"].index,    a["fc"].values/1e6,
            color=color, lw=2.2, linestyle="--", marker="o", ms=3,
            label=f"SARIMA Forecast (R²={a['r2']:.3f})")
    ax.axvline(a["train"].index[-1], color="gray", lw=1, linestyle=":")
    ax.set_title(f"Store {store} ({tier}) | RMSE=${a['rmse']:,.0f} | MAE=${a['mae']:,.0f}", fontsize=11)
    ax.set_ylabel("Sales (M$)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.1f}M"))
    ax.legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.show()

## CELL 17 — 12-WEEK FORECAST per Store (SARIMA)

In [ ]:
print("=" * 65)
print("SARIMA 12-WEEK FORECAST — Refit on Full Data per Store")
print("=" * 65)

forecast_store = {}
fig, axes = plt.subplots(5, 1, figsize=(14, 22), sharex=False)
fig.suptitle("SARIMA 12-Week Ahead Forecast — 5 Focus Stores\n"
             "(Shaded region = 95% Confidence Interval)",
             fontsize=14, fontweight="bold", y=1.01)

for idx, store in enumerate(FOCUS_STORES):
    if store not in sarima_results:
        print(f"  Store {store} — no SARIMA result, skipping forecast")
        continue
    ts = df[df["Store"] == store].sort_values("Date").set_index("Date")["Weekly_Sales"].copy()
    ts.index.freq = "W-FRI"
    color = list(STORE_COLORS.values())[idx]
    tier  = store_profile[store_profile["Store"]==store]["Sales_Tier"].values[0]

    # Refit on FULL series → best forecast
    full_fit = SARIMAX(
        ts,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 0, 52),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    fc_obj  = full_fit.get_forecast(steps=12)
    fc_mean = fc_obj.predicted_mean
    ci      = fc_obj.conf_int(alpha=0.05)

    forecast_store[store] = {
        "dates":  fc_mean.index.strftime("%Y-%m-%d").tolist(),
        "values": fc_mean.round(2).tolist(),
        "lower":  ci.iloc[:, 0].round(2).tolist(),
        "upper":  ci.iloc[:, 1].round(2).tolist(),
    }

    # Print table
    print(f"\n  Store {store:2d} ({tier}) — 12-Week Forecast:")
    print(f"  {'Week':>5} | {'Date':>12} | {'Predicted':>14} | {'Lower 95%CI':>14} | {'Upper 95%CI':>14}")
    print(f"  {'-'*66}")
    for i, (d, v, lo, hi) in enumerate(zip(
            fc_mean.index, fc_mean.values,
            ci.iloc[:,0].values, ci.iloc[:,1].values)):
        print(f"  {i+1:>5} | {str(d.date()):>12} | ${v:>13,.2f} | ${lo:>13,.2f} | ${hi:>13,.2f}")

    # Plot
    ax = axes[idx]
    last_52 = ts.iloc[-52:]
    ax.plot(last_52.index, last_52.values/1e6,
            color="steelblue", lw=1.8, label="Historical (last 52 wks)")
    ax.plot(fc_mean.index, fc_mean.values/1e6,
            color=color, lw=2.2, linestyle="--", marker="o", ms=5,
            label="12-Wk Forecast")
    ax.fill_between(ci.index,
                    ci.iloc[:, 0].values/1e6,
                    ci.iloc[:, 1].values/1e6,
                    color=color, alpha=0.18, label="95% Confidence Interval")
    ax.axvline(ts.index[-1], color="gray", lw=1.2,
               linestyle=":", label="Forecast Start")
    sr = sarima_results[store]
    ax.set_title(
        f"Store {store} ({tier})  |  SARIMA(1,1,1)(1,1,0,52)  |  "
        f"Test R²={sr['r2']:.3f}  RMSE=${sr['rmse']:,.0f}",
        fontsize=11
    )
    ax.set_ylabel("Weekly Sales (M$)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.1f}M"))
    ax.legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.show()

## CELL 18 — Individual Store Forecast Plots

In [ ]:
for store in FOCUS_STORES:
    if store not in sarima_results:
        print(f"  Store {store} — skipped (no SARIMA result)")
        continue
    ts = df[df["Store"] == store].sort_values("Date").set_index("Date")["Weekly_Sales"].copy()
    ts.index.freq = "W-FRI"
    color = list(STORE_COLORS.values())[FOCUS_STORES.index(store)]
    tier  = store_profile[store_profile["Store"]==store]["Sales_Tier"].values[0]

    fc    = forecast_store[store]
    fc_dates  = pd.to_datetime(fc["dates"])
    fc_values = np.array(fc["values"])
    fc_lower  = np.array(fc["lower"])
    fc_upper  = np.array(fc["upper"])

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(ts.index, ts.values/1e6, color="steelblue", lw=1.8, label="Full Historical Data")
    ax.plot(fc_dates, fc_values/1e6, color=color, lw=2.2,
            linestyle="--", marker="o", ms=5, label="12-Week SARIMA Forecast")
    ax.fill_between(fc_dates, fc_lower/1e6, fc_upper/1e6,
                    color=color, alpha=0.2, label="95% Confidence Interval")
    ax.axvline(ts.index[-1], color="gray", lw=1.3,
               linestyle=":", label="Forecast Start")
    avg_fc = np.mean(fc_values)
    avg_hist = ts.mean()
    ax.axhline(avg_hist/1e6, color="orange", lw=1, linestyle="--", alpha=0.6,
               label=f"Historical Avg: ${avg_hist/1e6:.2f}M")
    ax.set_title(f"Store {store} ({tier}) — Full History + 12-Week SARIMA Forecast",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Weekly Sales (M$)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.2f}M"))
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f"  Store {store}: Forecast avg = ${avg_fc:,.0f}/wk vs historical avg = ${avg_hist:,.0f}/wk\n")

## CELL 19 — SARIMA Residual Diagnostics

In [ ]:
print("Residual Diagnostics — Store", FOCUS_STORES[0])
ref_store = FOCUS_STORES[0]

if ref_store in sarima_results:
    fit_ref = sarima_results[ref_store]["fit"]
    resid   = fit_ref.resid.dropna()
    n_resid = len(resid)

    # Compute safe lag count — avoids ValueError for short series
    safe_lags = max(1, min(10, n_resid // 2 - 2))
    print(f"  Residual length: {n_resid}  |  Using lags={safe_lags}")

    try:
        fig = fit_ref.plot_diagnostics(figsize=(14, 8), lags=safe_lags)
        fig.suptitle(f"SARIMA Residual Diagnostics — Store {ref_store}",
                     fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        # Fallback manual plot if built-in fails
        print(f"  Built-in diagnostics failed ({e}) — showing manual plots")
        import scipy.stats as stats
        fig, axes = plt.subplots(2, 2, figsize=(13, 8))
        axes[0,0].plot(resid.values, color="#3B82F6", lw=1.5)
        axes[0,0].axhline(0, color="red", lw=1, linestyle="--")
        axes[0,0].set_title("Residuals over Time")
        axes[0,1].hist(resid.values, bins=max(5, n_resid//3), color="#22C55E",
                       edgecolor="white", alpha=0.85)
        axes[0,1].set_title("Residual Histogram")
        osm, osr = stats.probplot(resid.values, fit=False)
        axes[1,0].plot(osm, osr, "o", ms=4, color="#F97316")
        axes[1,0].plot(osm, np.polyval(np.polyfit(osm, osr, 1), osm),
                       "r-", lw=1.5)
        axes[1,0].set_title("Q-Q Plot")
        from statsmodels.graphics.tsaplots import plot_acf as _acf
        _acf(resid, lags=safe_lags, ax=axes[1,1], title="ACF of Residuals")
        plt.suptitle(f"SARIMA Residual Diagnostics — Store {ref_store}",
                     fontsize=12, fontweight="bold")
        plt.tight_layout(); plt.show()
else:
    print(f"  Store {ref_store} not in sarima_results — skipping diagnostics")

print("\nInterpretation:")
print("  Residual plot : Should look random (no trend/seasonality left)")
print("  Histogram     : Should be roughly bell-shaped / Normal")
print("  Q-Q plot      : Points close to the line = Normal residuals = good fit")
print("  ACF residuals : No significant spikes = white noise = good SARIMA fit")


## CELL 20 — Feature Engineering for ML Models

In [ ]:
df_ml = df.copy()
df_ml["Year"]    = df_ml["Date"].dt.year
df_ml["Month"]   = df_ml["Date"].dt.month
df_ml["Week"]    = df_ml["Date"].dt.isocalendar().week.astype(int)
df_ml["Quarter"] = df_ml["Date"].dt.quarter
df_ml["DayOfYear"] = df_ml["Date"].dt.dayofyear

# Lag features — same store, previous weeks
df_ml["Sales_Lag1"] = df_ml.groupby("Store")["Weekly_Sales"].shift(1)
df_ml["Sales_Lag2"] = df_ml.groupby("Store")["Weekly_Sales"].shift(2)
df_ml["Sales_Roll4"] = df_ml.groupby("Store")["Weekly_Sales"].transform(
    lambda x: x.shift(1).rolling(4).mean()
)

df_ml.dropna(inplace=True)

# Outlier capping on target
Q1  = df_ml["Weekly_Sales"].quantile(0.25)
Q3  = df_ml["Weekly_Sales"].quantile(0.75)
IQR = Q3 - Q1
df_ml["Weekly_Sales"] = df_ml["Weekly_Sales"].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

FEATURES = ["Store", "Holiday_Flag", "Temperature", "Fuel_Price", "CPI",
            "Unemployment", "Year", "Month", "Week", "Quarter", "DayOfYear",
            "Sales_Lag1", "Sales_Lag2", "Sales_Roll4"]
TARGET = "Weekly_Sales"

X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"✅ Feature engineering done: {X.shape}")
print(f"   Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"   Features: {FEATURES}")

## CELL 21 — Train 5 ML Models & Compare

In [ ]:
print("=" * 65)
print("ML MODEL TRAINING — 5 Models on All-Store Data")
print("=" * 65)

ml_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression":  Ridge(alpha=1.0),
    "Decision Tree":     DecisionTreeRegressor(random_state=42),
    "Random Forest":     RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

ml_results = {}
for name, model in ml_models.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    cv_r2 = cross_val_score(model, X_train_sc, y_train, cv=5, scoring="r2", n_jobs=-1).mean()
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    ml_results[name] = {"rmse": rmse, "mae": mae, "r2": r2, "cv_r2": cv_r2, "model": model}
    print(f"\n  {name}")
    print(f"    RMSE  : ${rmse:>10,.2f}")
    print(f"    MAE   : ${mae:>10,.2f}")
    print(f"    R²    : {r2:.4f}")
    print(f"    CV R² : {cv_r2:.4f}")

best_ml_name = max(ml_results, key=lambda k: ml_results[k]["r2"])
print(f"\n  🏆 Best ML Model: {best_ml_name} (R²={ml_results[best_ml_name]['r2']:.4f})")

## CELL 22 — ML Model Comparison Charts

In [ ]:
model_names  = list(ml_results.keys())
model_rmse   = [ml_results[m]["rmse"]/1e3 for m in model_names]
model_r2     = [ml_results[m]["r2"]      for m in model_names]
model_cv     = [ml_results[m]["cv_r2"]   for m in model_names]
bar_colors   = ["#22C55E" if m == best_ml_name else "#94A3B8" for m in model_names]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].barh(model_names, model_rmse, color=bar_colors, alpha=0.9)
axes[0].set_title("RMSE (K$) — Lower is Better")
axes[0].set_xlabel("RMSE (K$)")
for i, v in enumerate(model_rmse):
    axes[0].text(v + 0.3, i, f"{v:.1f}", va="center", fontsize=9)

axes[1].barh(model_names, model_r2, color=bar_colors, alpha=0.9)
axes[1].set_title("R² Score — Higher is Better")
axes[1].set_xlabel("R²")
axes[1].set_xlim([0.9, 1.0])
for i, v in enumerate(model_r2):
    axes[1].text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

axes[2].barh(model_names, model_cv, color=bar_colors, alpha=0.9)
axes[2].set_title("5-Fold CV R² — Higher is Better")
axes[2].set_xlabel("CV R²")
axes[2].set_xlim([0.9, 1.0])
for i, v in enumerate(model_cv):
    axes[2].text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

plt.suptitle("ML Model Comparison — All Stores", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## CELL 23 — Hyperparameter Tuning (Best ML Model)

In [ ]:
print("=" * 60)
print(f"HYPERPARAMETER TUNING — {best_ml_name}")
print("=" * 60)

param_dist = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [3, 5, 7, None],
    "learning_rate":     [0.05, 0.1, 0.2],
    "subsample":         [0.7, 0.8, 1.0],
    "min_samples_split": [2, 5],
}

base = GradientBoostingRegressor(random_state=42)
search = RandomizedSearchCV(
    base, param_dist,
    n_iter=15, cv=3, scoring="r2",
    random_state=42, n_jobs=-1, verbose=0
)
search.fit(X_train_sc, y_train)
tuned = search.best_estimator_

tuned_preds = tuned.predict(X_test_sc)
tuned_rmse  = np.sqrt(mean_squared_error(y_test, tuned_preds))
tuned_mae   = mean_absolute_error(y_test, tuned_preds)
tuned_r2    = r2_score(y_test, tuned_preds)

print(f"\n  Best parameters: {search.best_params_}")
print(f"  RMSE  : ${tuned_rmse:>10,.2f}")
print(f"  MAE   : ${tuned_mae:>10,.2f}")
print(f"  R²    : {tuned_r2:.4f}")
print(f"  Improvement in RMSE: {(ml_results[best_ml_name]['rmse']-tuned_rmse)/ml_results[best_ml_name]['rmse']*100:.1f}%")

## CELL 24 — Actual vs Predicted + Residual Plots

In [ ]:
sample = np.random.choice(len(y_test), size=min(500, len(y_test)), replace=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test.values[sample]/1e6, tuned_preds[sample]/1e6,
                alpha=0.4, s=12, color="#3B82F6")
mn = min(y_test.min(), tuned_preds.min()) / 1e6
mx = max(y_test.max(), tuned_preds.max()) / 1e6
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect Prediction")
axes[0].set_xlabel("Actual Weekly Sales (M$)")
axes[0].set_ylabel("Predicted Weekly Sales (M$)")
axes[0].set_title(f"Actual vs Predicted — Tuned GBM\nR²={tuned_r2:.4f}  RMSE=${tuned_rmse:,.0f}")
axes[0].legend()

# Residuals
residuals = y_test.values - tuned_preds
axes[1].scatter(tuned_preds[sample]/1e6, residuals[sample]/1e6,
                alpha=0.4, s=12, color="#F97316")
axes[1].axhline(0, color="black", lw=1.2, linestyle="--")
axes[1].set_xlabel("Predicted Sales (M$)")
axes[1].set_ylabel("Residuals (M$)")
axes[1].set_title("Residual Plot — Random Scatter = Good Fit")

plt.tight_layout()
plt.show()

## CELL 25 — Feature Importance (GBM)

In [ ]:
fi = pd.Series(tuned.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#22C55E" if v > 0.1 else "#3B82F6" if v > 0.05 else "#94A3B8"
          for v in fi.values]
ax.barh(fi.index, fi.values, color=colors)
ax.set_title("Feature Importance — Tuned Gradient Boosting",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Importance Score")
for i, (idx, v) in enumerate(fi.items()):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
for feat, imp in fi.sort_values(ascending=False).head(5).items():
    print(f"  {feat:<22} : {imp:.4f}")

## CELL 26 — Per-Store ML Prediction vs SARIMA

In [ ]:
print("=" * 65)
print("STORE-LEVEL COMPARISON: GBM vs SARIMA per Focus Store")
print("=" * 65)

fig, axes = plt.subplots(len(FOCUS_STORES), 1, figsize=(14, 4*len(FOCUS_STORES)))

for idx, store in enumerate(FOCUS_STORES):
    ts_store = df_ml[df_ml["Store"] == store].copy()
    ts_store = ts_store.sort_values("Date")

    Xs_store = scaler.transform(ts_store[FEATURES])
    ml_preds = tuned.predict(Xs_store)

    ax = axes[idx]
    color = list(STORE_COLORS.values())[idx]
    tier  = store_profile[store_profile["Store"]==store]["Sales_Tier"].values[0]

    ax.plot(ts_store["Date"], ts_store["Weekly_Sales"]/1e6,
            color="steelblue", lw=1.8, label="Actual Sales", alpha=0.8)
    ax.plot(ts_store["Date"], ml_preds/1e6,
            color="#F97316", lw=1.5, linestyle="--", alpha=0.7, label="GBM Prediction")

    # Add SARIMA 12-week forecast
    fc = forecast_store[store]
    fc_dates  = pd.to_datetime(fc["dates"])
    fc_values = np.array(fc["values"])
    fc_lower  = np.array(fc["lower"])
    fc_upper  = np.array(fc["upper"])
    ax.plot(fc_dates, fc_values/1e6,
            color=color, lw=2.2, marker="o", ms=5,
            label="SARIMA 12-Wk Forecast")
    ax.fill_between(fc_dates, fc_lower/1e6, fc_upper/1e6,
                    color=color, alpha=0.18, label="95% CI")
    ax.axvline(ts_store["Date"].max(), color="gray", lw=1, linestyle=":")

    sr = sarima_results[store]
    ax.set_title(f"Store {store} ({tier}) | SARIMA R²={sr['r2']:.3f} | "
                 f"GBM fits all-store signal", fontsize=11)
    ax.set_ylabel("Sales (M$)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.1f}M"))
    ax.legend(fontsize=9, loc="upper left")

plt.suptitle("Per-Store: GBM Fitted Values + SARIMA 12-Week Forecast",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## CELL 27 — Final Summary Table

In [ ]:
common = [s for s in FOCUS_STORES if s in arima_results and s in sarima_results]

print("\n" + "=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)

print("\n── ML Models (All-Store GBM) ──────────────────────────────────────")
print(f"  {'Model':<22} | {'RMSE':>12} | {'MAE':>12} | {'R²':>8} | {'CV R²':>8}")
print("  " + "-"*65)
for name in ml_results:
    r = ml_results[name]
    marker = " ◄ BEST" if name == best_ml_name else ""
    print(f"  {name:<22} | ${r['rmse']:>10,.0f} | ${r['mae']:>10,.0f} | {r['r2']:>8.4f} | {r['cv_r2']:>8.4f}{marker}")
print(f"\n  Tuned GBM (RandomizedSearchCV):")
print(f"    RMSE = ${tuned_rmse:>10,.0f} | MAE = ${tuned_mae:>10,.0f} | R² = {tuned_r2:.4f}")

print("\n── Time Series Models (SARIMA per Focus Store) ────────────────────")
print(f"  {'Store':>7} | {'Tier':>4} | {'ARIMA Order':>12} | {'ARIMA R²':>9} | {'SARIMA R²':>9} | {'SARIMA RMSE':>12}")
print("  " + "-"*65)
for s in FOCUS_STORES:
    a  = arima_results[s]
    sr = sarima_results[s]
    tier = store_profile[store_profile["Store"]==s]["Sales_Tier"].values[0]
    print(f"  Store {s:2d} | {tier:>4} | {str(a['order']):>12} | {a['r2']:>9.4f} | {sr['r2']:>9.4f} | ${sr['rmse']:>10,.0f}")

print("\n── 12-Week Forecast Summary ───────────────────────────────────────")
for s in FOCUS_STORES:
    fc = forecast_store[s]
    avg_fc = np.mean(fc["values"])
    tier = store_profile[store_profile["Store"]==s]["Sales_Tier"].values[0]
    print(f"  Store {s:2d} ({tier}): avg forecast = ${avg_fc:>12,.0f}/week | "
          f"range {fc['dates'][0]} → {fc['dates'][-1]}")

print("\n✅  All done! Scroll up to see all plots and tables.")

## CELL 28 — Save Models (optional download)

In [ ]:
import joblib, json, os

os.makedirs("models", exist_ok=True)

# Save GBM + scaler
joblib.dump(tuned,   "models/gbm_tuned.pkl")
joblib.dump(scaler,  "models/scaler.pkl")
joblib.dump(FEATURES,"models/features.pkl")

# Save forecast data
with open("models/sarima_forecasts.json", "w") as f:
    json.dump({str(k): v for k, v in forecast_store.items()}, f, indent=2)

# Save store profile
store_profile.to_csv("models/store_profile.csv", index=False)

print("✅  Models saved:")
print("   models/gbm_tuned.pkl")
print("   models/scaler.pkl")
print("   models/features.pkl")
print("   models/sarima_forecasts.json")
print("   models/store_profile.csv")

# Optional: Download all saved files in Colab
# from google.colab import files
# for fname in ["models/gbm_tuned.pkl","models/scaler.pkl","models/sarima_forecasts.json","models/store_profile.csv"]:
#     files.download(fname)